In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(palette="deep")
sns.set_context("poster")

In [ ]:
df_latency = pd.read_csv("data/latency_metrics.csv")
seq_length_map = {'promoter_all': 300,
          'promoter_tata': 300,
          'promoter_no_tata': 300,
          'enhancers': 400, 'enhancers_types': 400,
          'splice_sites_all': 600, 'splice_sites_acceptors': 600,
          'splice_sites_donors': 600, 'H2AFZ': 1000, 'H3K27ac': 1000,
          'H3K27me3': 1000, 'H3K36me3': 1000, 'H3K4me1': 1000,
          'H3K4me2': 1000, 'H3K4me3': 1000, 'H3K9ac': 1000,
          'H3K9me3': 1000, 'H4K20me1': 1000, 'H3': 1000,
         'H4': 1000, 'H3K14ac': 1000, 'H3K79me3': 1000}

In [ ]:
tasks = [
    "H2AFZ",
    "H3K9ac",
    "H3K27ac",
    "H3K4me1",
    "H3K4me2",
    "H3K4me3",
    "H3K27me3",
    "H3K36me3",
    "H3K9me3",
    "H4K20me1",
    "enhancers",
    "enhancers_types",
    "promoter_all",
    "promoter_tata",
    "promoter_no_tata",
    "splice_sites_donors", 
    "splice_sites_all", 
    "splice_sites_acceptors"
]

In [ ]:
df_latency.rename(index={i: df_latency['task_name'][i] for i in range(len(tasks))}, inplace=True)

In [ ]:
df_latency

In [ ]:
df_latency['Length'] = df_latency['task_name'].map(lambda x: seq_length_map.get(x, 200))

df_latency.rename(
    columns={'teacher_total_time_ms': 'Teacher Total Time (ms)',
             'student_total_time_ms': 'Student Total Time (ms)',
             "speedup_factor": "Speedup"},
    inplace=True
  )

In [ ]:
fig, ax = plt.subplots(figsize=(8,6))
ax.plot(tasks, df_latency['Teacher Total Time (ms)'][tasks], label='Teacher Model', marker='o')
ax.plot(tasks, df_latency['Student Total Time (ms)'][tasks], label='Distilled Model', marker='o')
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yscale('log')
ax.set_ylabel('Total Inference Time (ms)', fontsize=10)
plt.legend(fontsize=10)
plt.show()

In [ ]:
cmap = sns.color_palette("Blues", as_cmap=True)
fig = sns.relplot(
    df_latency,
    x='Teacher Total Time (ms)',
    y='Student Total Time (ms)',
    height=7.5,
    hue='Speedup',
    size='Length',
    palette=cmap
)

fig.ax.axline(xy1=(0, 0), slope=1, color='gray', linestyle='--')
fig.set(xscale="log", yscale="log")
fig.ax.set_xlim(10, 10000)
fig.ax.set_ylim(10, 10000)

# texts = []
# x_data = df_latency['Teacher Total Time (ms)']
# y_data = df_latency['Student Total Time (ms)']
# for i, task in enumerate(df_latency['task_name']):
#     texts.append(fig.ax.text(x_data.iloc[i], y_data.iloc[i], task.replace("_", " ").title(), fontsize=9))

# # Use adjust_text to prevent labels from overlapping
# adjust_text(texts, ax=fig.ax, arrowprops=dict(arrowstyle='-', color='black', lw=0.5))

plt.tight_layout()
#plt.savefig("output/latency.pdf", bbox_inches="tight")
plt.show()